In [ ]:
import pandas as pd

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [ ]:
!pip install apscheduler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.9 MB/s eta 0:00:00


In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
import numpy as np
import datetime
import sys, subprocess, re
import time

In [ ]:
sh_base = gc.open('ЦД1').sheet1
sh = gc.open('Data Mart').sheet1

In [ ]:
# ---- параметры ----
SPREADSHEET_NAME = 'ЦД1'        # исходные данные
DATAMART_NAME = 'Data Mart'     # витрина
DATAMART_SHEET = 'Sheet1'       # если в Data Mart не первый лист, замените

# Открываем основную таблицу
wb = gc.open(SPREADSHEET_NAME)
# Получаем листы по точным названиям
money_sheet = wb.worksheet('Деньги')
clients_sheet = wb.worksheet('Клиенты')

# Открываем таблицу-датамарт и берём первый лист
dm_wb = gc.open(DATAMART_NAME)
datamart = dm_wb.sheet1

In [ ]:
# логика вычислений KPI и записи в Data Mart
def calculate_kpi_and_write():
    try:
        # --- читаем данные ---
        df_money = safe_read_sheet(money_sheet)

        # обнаруживаем колонки (money)
        m_date_col = 'Дата'
        m_expenses_col = 'Расходы'
        m_sales_col = 'Объем продаж'

        # текущая дата, начало месяца
        now = datetime.datetime.now()
        start_month = datetime.datetime(now.year, now.month, 1)

        # --- агрегируем деньги ---
        # Преобразуем дату и фильтруем по текущему месяцу
        df_money[m_date_col] = pd.to_datetime(df_money[m_date_col], errors='coerce')
        df_money = df_money[df_money[m_date_col] >= start_month]

        # Преобразуем в числа и суммируем
        df_money[m_expenses_col] = pd.to_numeric(df_money[m_expenses_col], errors='coerce').fillna(0)
        df_money[m_sales_col] = pd.to_numeric(df_money[m_sales_col], errors='coerce').fillna(0)

        # Считаем суммы
        expenses_sum = int(df_money[m_expenses_col].sum())
        sales_sum = int(df_money[m_sales_col].sum())

        # Прибыль
        profit = (sales_sum * 15000) - expenses_sum

        # Здесь должен быть код для записи в Data Mart
        # Например:
        # datamart.update('A1', [[profit]])

    except Exception as e:
        print(f"Ошибка: {e}")

In [ ]:
#Симплекс-метод
from scipy.optimize import linprog

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Количество рабочих дней
working_days = 22

# --- ДНЕВНЫЕ ОГРАНИЧЕНИЯ ---
profit_per_sale = 15000
staff_per_sale = 0.5
max_staff = 10
calls_per_sale = 10
max_calls_per_staff = 50
max_sales = 25

# Решаем ДНЕВНУЮ задачу
obj_day = [-profit_per_sale]
lhs_ineq_day = [
    [staff_per_sale],
    [calls_per_sale],
    [1],
    [-1]
]
rhs_ineq_day = [
    max_staff,
    max_staff * max_calls_per_staff,
    max_sales,
    0
]

opt_day = linprog(c=obj_day, A_ub=lhs_ineq_day, b_ub=rhs_ineq_day, method="revised simplex")
optimal_sales_day = opt_day.x[0]
max_profit_day = -opt_day.fun

# МЕСЯЧНЫЕ показатели = ДНЕВНЫЕ * working_days
optimal_sales_month = optimal_sales_day * working_days
max_profit_month = max_profit_day * working_days

print("=" * 60)
print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ")
print("=" * 60)
print(f"\nДневные показатели:")
print(f"  Оптимальный объем продаж: {optimal_sales_day:.0f} ед./день")
print(f"  Максимальная прибыль: {max_profit_day:,.0f} руб./день")

print(f"\nМесячные показатели (при {working_days} рабочих днях):")
print(f"  Оптимальный объем продаж: {optimal_sales_month:.0f} ед./мес")
print(f"  Максимальная прибыль: {max_profit_month:,.0f} руб./мес")

РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ

Дневные показатели:
  Оптимальный объем продаж: 20 ед./день
  Максимальная прибыль: 300,000 руб./день

Месячные показатели (при 22 рабочих днях):
  Оптимальный объем продаж: 440 ед./мес
  Максимальная прибыль: 6,600,000 руб./мес


/tmp/ipykernel_3532/1428975389.py:30: DeprecationWarning: `method='revised simplex'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  opt_day = linprog(c=obj_day, A_ub=lhs_ineq_day, b_ub=rhs_ineq_day, method="revised simplex")


In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from google.api_core import retry
from time import sleep
creds, _ = default()
gc = gspread.authorize(creds)

import pandas as pd
import datetime
import time
import random
from apscheduler.schedulers.background import BackgroundScheduler

# Открываем основную таблицу
SPREADSHEET_NAME = 'ЦД1'
sh = gc.open(SPREADSHEET_NAME)

# Получаем листы
money_sheet = sh.worksheet('Деньги')
staff_sheet = sh.worksheet('Сотрудники')
clients_sheet = sh.worksheet('Клиенты')

# Открываем таблицу Data Mart
DATAMART_NAME = 'Data Mart'
dm_wb = gc.open(DATAMART_NAME)
datamart = dm_wb.sheet1

# Helper function to safely read data from a GSheet
def safe_read_sheet(sheet):
    try:
        data = sheet.get_all_values()
        if not data:
            return pd.DataFrame()
        columns = data[0]
        df = pd.DataFrame(data[1:], columns=columns)
        return df
    except Exception as e:
        print(f"Error reading sheet {sheet.title}: {e}")
        return pd.DataFrame()

# Функция для записи с повторными попытками
def update_sheet_with_retry(sheet, data, max_retries=3):
    for attempt in range(max_retries):
        try:
            sheet.clear()
            sheet.update('A1', data)
            return True
        except Exception as e:
            if "429" in str(e) or "Quota" in str(e):
                wait_time = (attempt + 1) * 5  # 5, 10, 15 секунд
                print(f"Квота превышена, ждем {wait_time} секунд...")
                time.sleep(wait_time)
            else:
                print(f"Ошибка при записи: {e}")
                return False
    return False

# логика вычислений KPI и записи в Data Mart
def calculate_kpi_and_write():
    try:
        print(f"\n[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Начинаем расчет...")

        # --- читаем данные ---
        df_money = safe_read_sheet(money_sheet)
        df_clients = safe_read_sheet(clients_sheet)
        df_staff = safe_read_sheet(staff_sheet)

        # обнаруживаем колонки (money)
        m_date_col = 'Дата'
        m_expenses_col = 'Расходы'
        m_sales_col = 'Объем продаж'

        # Преобразуем дату
        if not df_money.empty:
            df_money[m_date_col] = pd.to_datetime(df_money[m_date_col], errors='coerce')

        # Определяем последний месяц, за который есть данные
        if not df_money.empty:
            # Находим максимальную дату в данных
            max_date = df_money[m_date_col].max()
            # Определяем начало последнего месяца в данных
            start_month = datetime.datetime(max_date.year, max_date.month, 1)

            # Определяем конец последнего месяца
            if max_date.month == 12:
                end_month = datetime.datetime(max_date.year + 1, 1, 1) - datetime.timedelta(days=1)
            else:
                end_month = datetime.datetime(max_date.year, max_date.month + 1, 1) - datetime.timedelta(days=1)

            print(f"Анализируем данные за {start_month.strftime('%B %Y')}")
            print(f"Период: {start_month.strftime('%d.%m.%Y')} - {end_month.strftime('%d.%m.%Y')}")
        else:
            start_month = datetime.datetime.now().replace(day=1)
            print("Данные пусты, использую текущий месяц")

        # --- агрегируем деньги и продажи ---
        if not df_money.empty:
            # Фильтруем по последнему месяцу
            df_money_month = df_money[df_money[m_date_col] >= start_month].copy()

            # Преобразуем в числа и суммируем
            df_money_month[m_expenses_col] = pd.to_numeric(df_money_month[m_expenses_col], errors='coerce').fillna(0)
            df_money_month[m_sales_col] = pd.to_numeric(df_money_month[m_sales_col], errors='coerce').fillna(0)

            # Считаем суммы
            expenses_sum = int(df_money_month[m_expenses_col].sum())
            sales_sum = int(df_money_month[m_sales_col].sum())

            # Считаем количество продаж
            sales_count = len(df_money_month)
            sales_volume = sales_sum

            print(f"Найдено записей за месяц: {sales_count}")
            print(f"Сумма продаж: {sales_volume}")

        else:
            expenses_sum = 0
            sales_sum = 0
            sales_count = 0
            sales_volume = 0
            print("Предупреждение: данные по деньгам пусты")

        # Прибыль (прибыль от одной продажи 15 000 руб)
        profit = (sales_sum * 15000) - expenses_sum

        # --- агрегируем клиентов ---
        c_date_col = 'Дата'
        c_finished_col = 'Количество завершенных договоров'
        c_new_col = 'Количество новых договоров'

        if not df_clients.empty:
            # Преобразуем дату
            df_clients[c_date_col] = pd.to_datetime(df_clients[c_date_col], errors='coerce')

            # Находим максимальную дату в данных клиентов
            max_date_clients = df_clients[c_date_col].max()
            start_month_clients = datetime.datetime(max_date_clients.year, max_date_clients.month, 1)

            # Фильтруем по последнему месяцу
            df_clients_month = df_clients[df_clients[c_date_col] >= start_month_clients].copy()

            # Преобразуем в числа и суммируем
            df_clients_month[c_finished_col] = pd.to_numeric(df_clients_month[c_finished_col], errors='coerce').fillna(0)
            df_clients_month[c_new_col] = pd.to_numeric(df_clients_month[c_new_col], errors='coerce').fillna(0)

            # Считаем суммы за месяц
            finished_sum = int(df_clients_month[c_finished_col].sum())
            new_sum = int(df_clients_month[c_new_col].sum())

        else:
            finished_sum = 0
            new_sum = 0
            print("Предупреждение: данные по клиентам пусты")

        # --- обрабатываем данные по сотрудникам ---
        total_staff = 12

        if not df_staff.empty and 'Количество работающих сотрудников' in df_staff.columns:
            df_staff['Количество работающих сотрудников'] = pd.to_numeric(df_staff['Количество работающих сотрудников'], errors='coerce').fillna(0)
            last_staff = int(df_staff['Количество работающих сотрудников'].iloc[-1])
            non_working_staff = total_staff - last_staff
        else:
            last_staff = 0
            non_working_staff = total_staff
            print("Предупреждение: данные по сотрудникам не найдены")

        # --- вычисляем цветовые коды ---
        # Прибыль
        if profit < 2000000:
            profit_color = 0
        elif profit <= 5500000:
            profit_color = 1
        else:
            profit_color = 2

        # Объем продаж
        if sales_sum < 250:
            sales_sum_color = 0
        elif sales_sum <= 400:
            sales_sum_color = 1
        else:
            sales_sum_color = 2

        # Работающие сотрудники
        if last_staff < 6:
            last_staff_color = 0
        elif last_staff < 8:
            last_staff_color = 1
        else:
            last_staff_color = 2

        # Неработающие сотрудники
        if non_working_staff < 3:
            non_working_staff_color = 2
        elif non_working_staff < 6:
            non_working_staff_color = 1
        else:
            non_working_staff_color = 0

        # Завершенные договоры
        if finished_sum < 5:
            finished_color = 2
        elif finished_sum < 10:
            finished_color = 1
        else:
            finished_color = 0

        # Новые договоры
        if new_sum < 5:
            new_color = 0
        elif new_sum < 10:
            new_color = 1
        else:
            new_color = 2

        rows = [
            ['Показатель', 'Значение', 'Цвет'],
            ['Прибыль за месяц', profit, profit_color],
            ['Объем продаж (количество) за месяц', sales_sum, sales_sum_color],
            ['Количество завершенных договоров за месяц', finished_sum, 4],
            ['Количество договоров в работе за месяц', new_sum, new_color],
            ['Количество работающих сотрудников (последнее значение)', last_staff, last_staff_color],
            ['Количество неработающих сотрудников', non_working_staff, non_working_staff_color],
            ['Оптимальный объем продаж в месяц', '440', '2'],
            ['Максимальная прибыль в месяц', '6600000', '2']
        ]

        # очищаем и записываем с повторными попытками
        if update_sheet_with_retry(datamart, rows):
            print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Витрина обновлена.")
            print(f"Продажи за месяц: {sales_count} шт. на сумму {sales_volume:,.0f} руб.")
            print(f"Сотрудники: работающих - {last_staff}, неработающих - {non_working_staff}")
            print(f"Договоры: новых - {new_sum}, завершенных - {finished_sum}")
        else:
            print("Не удалось обновить витрину после нескольких попыток")

    except Exception as e:
        print("Ошибка в calculate_kpi_and_write():", str(e))

# первый запуск
calculate_kpi_and_write()

# Увеличиваем интервал обновления до 60 секунд
scheduler = BackgroundScheduler()
scheduler.add_job(calculate_kpi_and_write, 'interval', seconds=60)  # 60 секунд вместо 10
scheduler.start()

print("Автообновление запущено (каждые 60 секунд)")

# чтобы Colab не завершил выполнение
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Остановлено вручную.")
    scheduler.shutdown()


[2026-04-15 12:27:53] Начинаем расчет...
Анализируем данные за April 2026
Период: 01.04.2026 - 30.04.2026
Найдено записей за месяц: 1
Сумма продаж: 76


/tmp/ipykernel_7952/1418647524.py:49: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update('A1', data)


[2026-04-15 12:27:53] Витрина обновлена.
Продажи за месяц: 1 шт. на сумму 76 руб.
Сотрудники: работающих - 10, неработающих - 2
Договоры: новых - 7, завершенных - 3
Автообновление запущено (каждые 60 секунд)

[2026-04-15 12:28:53] Начинаем расчет...
Анализируем данные за April 2026
Период: 01.04.2026 - 30.04.2026
Найдено записей за месяц: 1
Сумма продаж: 76


/tmp/ipykernel_7952/1418647524.py:49: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update('A1', data)


[2026-04-15 12:28:55] Витрина обновлена.
Продажи за месяц: 1 шт. на сумму 76 руб.
Сотрудники: работающих - 10, неработающих - 2
Договоры: новых - 7, завершенных - 3

[2026-04-15 12:29:53] Начинаем расчет...
Анализируем данные за April 2026
Период: 01.04.2026 - 30.04.2026
Найдено записей за месяц: 1
Сумма продаж: 76


...

[2026-04-15 14:24:55] Витрина обновлена.
Продажи за месяц: 2 шт. на сумму 293 руб.
Сотрудники: работающих - 10, неработающих - 2
Договоры: новых - 9, завершенных - 4

[2026-04-15 14:25:53] Начинаем расчет...
Анализируем данные за April 2026
Период: 01.04.2026 - 30.04.2026
Найдено записей за месяц: 2
Сумма продаж: 293


/tmp/ipykernel_7952/1418647524.py:49: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update('A1', data)


Квота превышена, ждем 5 секунд...
Квота превышена, ждем 10 секунд...
Квота превышена, ждем 15 секунд...
Не удалось обновить витрину после нескольких попыток

[2026-04-15 14:26:53] Начинаем расчет...
Анализируем данные за April 2026
Период: 01.04.2026 - 30.04.2026
Найдено записей за месяц: 2
Сумма продаж: 293


/tmp/ipykernel_7952/1418647524.py:49: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update('A1', data)


[2026-04-15 14:26:54] Витрина обновлена.
Продажи за месяц: 2 шт. на сумму 293 руб.
Сотрудники: работающих - 10, неработающих - 2
Договоры: новых - 9, завершенных - 4


In [ ]:
# ============== ОСНОВНОЙ КОД (вызов классификатора) ==============

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

import datetime
import sys
import os

# Монтируем Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Добавляем путь к модулю
module_path = '/content/drive/MyDrive/Colab Notebooks/ЦД/ЦД1/'
if module_path not in sys.path:
    sys.path.insert(0, module_path)

print(f"📁 Путь к модулям: {module_path}")

# Проверяем, есть ли файл
if os.path.exists(os.path.join(module_path, 'PMOV.py')):
    print("✅ Файл PMOV.py найден")
else:
    print("❌ Файл PMOV.py не найден!")
    print("Содержимое папки:", os.listdir(module_path) if os.path.exists(module_path) else "Папка не существует")

# Импортируем функции из классификатора
from PMOV import get_prediction_for_all_clients

# Открываем таблицу Data Mart
sh = gc.open('Data Mart')

# Получаем лист Data Mart
try:
    datamart = sh.worksheet('Data Mart')
    datamart.clear()  # Очищаем лист перед записью
    print("✅ Лист Data Mart найден и очищен")
except:
    datamart = sh.add_worksheet('Data Mart', rows=1000, cols=10)
    print("✅ Лист Data Mart создан")

# ============== ПОЛУЧАЕМ ПРОГНОЗЫ ДЛЯ ВСЕХ КЛИЕНТОВ ==============
print("\n🔄 Получение прогнозов для всех клиентов...")

try:
    # Получаем DataFrame с прогнозами для всех клиентов
    all_predictions = get_prediction_for_all_clients()

    print(f"✅ Получены прогнозы для {len(all_predictions)} клиентов")

    # ============== ЗАПИСЬ В DATA MART ==============
    # Заголовки таблицы
    headers = ['ID клиента', 'Вероятность заключения договора']
    datamart.update('A1', [headers])

    # Подготавливаем данные для записи
    rows_to_write = []
    for _, row in all_predictions.iterrows():
        rows_to_write.append([
            int(row['ID клиента']),
            f"{row['Вероятность_заключения']}%"
        ])

    # Записываем данные
    if rows_to_write:
        datamart.update('A2', rows_to_write)
        print(f"✅ Записано {len(rows_to_write)} строк в лист 'Data Mart'")

    # ============== ВЫВОД СТАТИСТИКИ В КОНСОЛЬ ==============
    print("\n" + "=" * 60)
    print("СТАТИСТИКА ПРОГНОЗОВ")
    print("=" * 60)
    print(f"\n📊 Всего клиентов: {len(all_predictions)}")
    print(f"📈 Средняя вероятность: {all_predictions['Вероятность_заключения'].mean():.1f}%")
    print(f"📈 Максимальная вероятность: {all_predictions['Вероятность_заключения'].max():.1f}%")
    print(f"📉 Минимальная вероятность: {all_predictions['Вероятность_заключения'].min():.1f}%")

    # Топ-5 клиентов
    print("\n🏆 ТОП-5 клиентов с наибольшей вероятностью:")
    top_clients = all_predictions.nlargest(5, 'Вероятность_заключения')
    for _, row in top_clients.iterrows():
        print(f"   ID {int(row['ID клиента'])}: {row['Вероятность_заключения']}%")

    # Клиенты с низкой вероятностью
    low_clients = all_predictions[all_predictions['Вероятность_заключения'] < 30]
    print(f"\n⚠️ Клиентов с вероятностью < 30%: {len(low_clients)}")

    # Распределение по уровням вероятности
    high = len(all_predictions[all_predictions['Вероятность_заключения'] >= 70])
    medium = len(all_predictions[(all_predictions['Вероятность_заключения'] >= 40) & (all_predictions['Вероятность_заключения'] < 70)])
    low = len(all_predictions[all_predictions['Вероятность_заключения'] < 40])

    print(f"\n📊 Распределение по уровням вероятности:")
    print(f"   Высокая (≥70%): {high} клиентов")
    print(f"   Средняя (40-69%): {medium} клиентов")
    print(f"   Низкая (<40%): {low} клиентов")

except Exception as e:
    print(f"❌ Ошибка при получении прогнозов: {e}")

print("\n" + "=" * 60)
print("✅ ГОТОВО!")
print("=" * 60)
print("\n📋 В лист 'Data Mart' записаны столбцы:")
print("   1. ID клиента")
print("   2. Вероятность заключения договора (%)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Путь к модулям: /content/drive/MyDrive/Colab Notebooks/ЦД/ЦД1/
✅ Файл PMOV.py найден
🔄 Обучение классификатора на данных клиентов...
   - Загружено записей: 100
✅ Классификатор обучен!
   - Признаков в модели: 14
   - Средняя точность: 100.00%
   - Обучающих примеров: 100

📊 Топ-5 важных признаков:
   Стадия_Договор заключен: 0.523
   Потребность_уровень: 0.163
   Стадия_Работа над договором: 0.132
   Постоянный: 0.054
   Стадия_Договор расторгнут: 0.051
✅ Модуль PMOV загружен и готов к использованию
   Доступные функции:
   - get_prediction_for_client(client_data) - прогноз для одного клиента
   - get_prediction_for_all_clients() - прогноз для всех клиентов
   - get_prediction_today() - прогноз для последнего клиента
✅ Лист Data Mart найден и очищен

🔄 Получение прогнозов для всех клиентов...
✅ Получены прогнозы для 100 клиентов


/tmp/ipykernel_3532/68667412.py:60: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  datamart.update('A1', [headers])
/tmp/ipykernel_3532/68667412.py:72: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  datamart.update('A2', rows_to_write)


✅ Записано 100 строк в лист 'Data Mart'

СТАТИСТИКА ПРОГНОЗОВ

📊 Всего клиентов: 100
📈 Средняя вероятность: 37.2%
📈 Максимальная вероятность: 100.0%
📉 Минимальная вероятность: 0.0%

🏆 ТОП-5 клиентов с наибольшей вероятностью:
   ID 3: 100.0%
   ID 4: 100.0%
   ID 8: 100.0%
   ID 11: 100.0%
   ID 13: 100.0%

⚠️ Клиентов с вероятностью < 30%: 63

📊 Распределение по уровням вероятности:
   Высокая (≥70%): 37 клиентов
   Средняя (40-69%): 0 клиентов
   Низкая (<40%): 63 клиентов

✅ ГОТОВО!

📋 В лист 'Data Mart' записаны столбцы:
   1. ID клиента
   2. Вероятность заключения договора (%)


In [ ]:
# -*- coding: utf-8 -*-

import math
import heapq
from collections import deque

import numpy as np
import pandas as pd
import gspread

from google.colab import auth
from google.auth import default

# =========================================================
# НАСТРОЙКИ
# =========================================================
SPREADSHEET_NAME = "ЦД1"
INPUT_SHEET_NAME = "Сотрудники"
OUTPUT_SHEET_NAME = "результаты СМО"

CURRENT_STAFF = 3                  # штат фиксирован: 3 сотрудника
MAX_STAFF_FOR_RECOMMENDATION = 10  # до скольких сотрудников проверять рекомендацию
INTERVAL_MINUTES = 10              # интервал для пересчета потока
DEFAULT_SERVICE_SECONDS = 180      # средняя длительность звонка, сек
DEFAULT_MAX_WAIT_SECONDS = 60      # максимальное время ожидания, сек
SIM_HORIZON_HOURS = 8              # горизонт моделирования
REPLICATIONS = 300                 # число прогонов для усреднения
RANDOM_SEED = 42

# =========================================================
# ПОДКЛЮЧЕНИЕ К GOOGLE SHEETS
# =========================================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)

# =========================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# =========================================================
def read_last_numeric(df: pd.DataFrame, col: str, default_value: float):
    if col not in df.columns:
        return default_value
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if s.empty:
        return default_value
    return float(s.iloc[-1])

def simulate_mm_m_with_patience(
    lambda_per_min: float,
    mu_per_min: float,
    m: int,
    max_wait_seconds: float,
    horizon_hours: float = 8.0,
    replications: int = 200,
    seed: int = 42,
):
    """
    Имитация M/M/m с ограничением по времени ожидания.
    Если клиент не дождался свободного сотрудника за max_wait_seconds,
    он считается отказом.
    """
    base_rng = np.random.default_rng(seed)
    horizon_sec = horizon_hours * 3600.0
    stats = []

    for _ in range(replications):
        local_seed = int(base_rng.integers(0, 10**9))
        local_rng = np.random.default_rng(local_seed)

        t = 0.0
        next_arrival = local_rng.exponential(1.0 / lambda_per_min) * 60.0

        busy = []          # времена окончания обслуживания
        queue = deque()    # (arrival_time, deadline_time)

        arrivals = 0
        served = 0
        abandoned = 0
        waited_served = 0

        sum_wait_served = 0.0
        sum_service_time = 0.0

        area_queue = 0.0
        area_busy = 0.0
        area_empty = 0.0
        last_t = 0.0

        def advance_time(new_t):
            nonlocal last_t, area_queue, area_busy, area_empty
            dt = new_t - last_t
            if dt < 0:
                return
            area_queue += len(queue) * dt
            area_busy += len(busy) * dt
            if len(queue) == 0 and len(busy) == 0:
                area_empty += dt
            last_t = new_t

        def remove_expired(current_time):
            nonlocal abandoned
            while queue and queue[0][1] <= current_time:
                queue.popleft()
                abandoned += 1

        while True:
            next_completion = busy[0] if busy else float("inf")
            next_timeout = queue[0][1] if queue else float("inf")
            next_event = min(next_arrival, next_completion, next_timeout)

            if next_event == float("inf"):
                break

            advance_time(next_event)
            t = next_event

            if queue and queue[0][1] <= t and next_event == next_timeout:
                remove_expired(t)
                continue

            if busy and next_event == next_completion:
                heapq.heappop(busy)
                remove_expired(t)

                if queue:
                    arrival_time, _ = queue.popleft()
                    wait_time = t - arrival_time
                    served += 1
                    waited_served += 1
                    sum_wait_served += wait_time

                    service_time = local_rng.exponential(1.0 / mu_per_min) * 60.0
                    sum_service_time += service_time
                    heapq.heappush(busy, t + service_time)
                continue

            if next_event == next_arrival:
                arrivals += 1
                remove_expired(t)

                if len(busy) < m:
                    served += 1
                    service_time = local_rng.exponential(1.0 / mu_per_min) * 60.0
                    sum_service_time += service_time
                    heapq.heappush(busy, t + service_time)
                else:
                    deadline = t + max_wait_seconds
                    queue.append((t, deadline))

                interarrival = local_rng.exponential(1.0 / lambda_per_min) * 60.0
                next_arrival = t + interarrival
                if next_arrival > horizon_sec:
                    next_arrival = float("inf")
                continue

        total_time = max(last_t, 1e-9)

        p_otk = abandoned / arrivals if arrivals > 0 else 0.0
        p_wait = waited_served / arrivals if arrivals > 0 else 0.0
        p_pr = area_empty / total_time
        lq = area_queue / total_time
        ls = area_busy / total_time
        wq_sec = sum_wait_served / served if served > 0 else 0.0
        w_sec = (sum_wait_served + sum_service_time) / served if served > 0 else 0.0
        utilization_per_staff = ls / m if m > 0 else 0.0

        stats.append({
            "arrivals": arrivals,
            "served": served,
            "abandoned": abandoned,
            "p_otk": p_otk,
            "p_wait": p_wait,
            "p_pr": p_pr,
            "lq": lq,
            "ls": ls,
            "wq_sec": wq_sec,
            "w_sec": w_sec,
            "utilization_per_staff": utilization_per_staff,
        })

    df = pd.DataFrame(stats)
    return {
        "arrivals_mean": df["arrivals"].mean(),
        "served_mean": df["served"].mean(),
        "abandoned_mean": df["abandoned"].mean(),
        "P_otk": df["p_otk"].mean(),
        "P_wait": df["p_wait"].mean(),
        "P_pr": df["p_pr"].mean(),
        "Lq": df["lq"].mean(),
        "Ls": df["ls"].mean(),
        "Wq_sec": df["wq_sec"].mean(),
        "W_sec": df["w_sec"].mean(),
        "utilization_per_staff": df["utilization_per_staff"].mean(),
    }

def find_recommended_staff(lambda_per_min: float, mu_per_min: float, max_wait_seconds: float):
    """
    Подбор числа сотрудников:
    ищем минимальное m, при котором P_otk <= 5% и Wq <= 60 сек.
    """
    for m in range(1, MAX_STAFF_FOR_RECOMMENDATION + 1):
        res = simulate_mm_m_with_patience(
            lambda_per_min=lambda_per_min,
            mu_per_min=mu_per_min,
            m=m,
            max_wait_seconds=max_wait_seconds,
            horizon_hours=SIM_HORIZON_HOURS,
            replications=max(100, REPLICATIONS // 2),
            seed=RANDOM_SEED + m,
        )
        if res["P_otk"] <= 0.05 and res["Wq_sec"] <= 60:
            return m
    return MAX_STAFF_FOR_RECOMMENDATION

def get_worksheet_by_possible_names(spreadsheet, names):
    for name in names:
        try:
            return spreadsheet.worksheet(name)
        except Exception:
            pass
    return None

# =========================================================
# ЧТЕНИЕ ДАННЫХ ИЗ ЛИСТА "Сотрудники"
# =========================================================
try:
    staff_ws = get_worksheet_by_possible_names(sh, [INPUT_SHEET_NAME, INPUT_SHEET_NAME.lower()])
    if staff_ws is None:
        raise ValueError(f"Не найден лист '{INPUT_SHEET_NAME}'")

    values = staff_ws.get_all_values()
    if len(values) <= 1:
        raise ValueError(f"Лист '{staff_ws.title}' пустой или содержит только заголовок.")

    df_staff = pd.DataFrame(values[1:], columns=values[0])

    calls_per_hour = read_last_numeric(df_staff, "Количество звонков в час", 60)
    service_seconds = read_last_numeric(df_staff, "Средняя длительность звонка, сек", DEFAULT_SERVICE_SECONDS)
    max_wait_seconds = read_last_numeric(df_staff, "Максимальное время ожидания, сек", DEFAULT_MAX_WAIT_SECONDS)

except Exception as e:
    print(f"⚠️ Не удалось прочитать лист '{INPUT_SHEET_NAME}': {e}")
    calls_per_hour = 60
    service_seconds = DEFAULT_SERVICE_SECONDS
    max_wait_seconds = DEFAULT_MAX_WAIT_SECONDS

service_seconds = max(1.0, service_seconds)
max_wait_seconds = max(1.0, max_wait_seconds)

# =========================================================
# ПЕРЕВОД В ИНТЕНСИВНОСТИ
# =========================================================
lambda_per_min = calls_per_hour / 60.0
lambda_per_10min = lambda_per_min * INTERVAL_MINUTES
mu_per_min = 60.0 / service_seconds

m = CURRENT_STAFF
rho = lambda_per_min / (m * mu_per_min)

# =========================================================
# РАСЧЕТ СМО С ОГРАНИЧЕНИЕМ ПО ВРЕМЕНИ ОЖИДАНИЯ
# =========================================================
res = simulate_mm_m_with_patience(
    lambda_per_min=lambda_per_min,
    mu_per_min=mu_per_min,
    m=m,
    max_wait_seconds=max_wait_seconds,
    horizon_hours=SIM_HORIZON_HOURS,
    replications=REPLICATIONS,
    seed=RANDOM_SEED,
)

P_pr = res["P_pr"]
P_otk = res["P_otk"]
P_wait = res["P_wait"]
Lq = res["Lq"]
Ls = res["Ls"]
Wq_sec = res["Wq_sec"]
W_sec = res["W_sec"]
utilization_per_staff = res["utilization_per_staff"]

lost_calls_per_hour = calls_per_hour * P_otk
lost_calls_per_10min = lambda_per_10min * P_otk

recommended_staff = find_recommended_staff(lambda_per_min, mu_per_min, max_wait_seconds)

# =========================================================
# РЕШЕНИЕ ДЛЯ ЛПР
# =========================================================
if recommended_staff > m:
    decision = f"УВЕЛИЧИТЬ ШТАТ до {recommended_staff} сотрудников"
elif recommended_staff < m:
    decision = f"СОКРАТИТЬ ШТАТ до {recommended_staff} сотрудников"
else:
    decision = "ШТАТ ОПТИМАЛЕН"

recommendations = [f"Решение: {decision}"]

if recommended_staff > m:
    if P_otk > 0.05:
        recommendations.append("Причина: высокая доля потерянных звонков")
    if Wq_sec > 60:
        recommendations.append("Причина: клиенты слишком долго ждут")
elif recommended_staff < m:
    recommendations.append("Причина: сотрудники недозагружены")
else:
    recommendations.append("Система работает эффективно")

if rho > 0.85:
    recommendations.append("Система близка к перегрузке")

# =========================================================
# ВЫВОД В ЛИСТ Data Mart "результаты СМО"
# =========================================================
try:
    out_ws = sh.worksheet(OUTPUT_SHEET_NAME)
    out_ws.clear()
except Exception:
    out_ws = sh.add_worksheet(title=OUTPUT_SHEET_NAME, rows=100, cols=10)

dashboard_rows = [
    ["Метрика", "Значение"],
    ["Количество сотрудников колл-центра, чел.", m],
    ["Звонков в час", round(calls_per_hour, 2)],
    ["Коэффициент загрузки ρ", round(rho, 4)],
    ["Среднее время ожидания, сек", round(Wq_sec, 2)],
    ["Вероятность ожидания", round(P_wait, 4)],
    ["Вероятность отказа", round(P_otk, 6)],
    ["Потерянные звонки в час", round(lost_calls_per_hour, 2)],
    ["Средняя длина очереди", round(Lq, 2)],
    ["Рекомендуемый штат, чел.", recommended_staff],
]

for rec in recommendations:
    dashboard_rows.append([rec, "", ""])

out_ws.update("A1", dashboard_rows)

# =========================================================
# КОНСОЛЬНЫЙ ВЫВОД
# =========================================================
print("=" * 70)
print("СИСТЕМА МАССОВОГО ОБСЛУЖИВАНИЯ С ОГРАНИЧЕНИЕМ ПО ВРЕМЕНИ ОЖИДАНИЯ")
print("=" * 70)
print(f"Сотрудников: {m}")
print(f"Звонков в час: {calls_per_hour}")
print(f"Средняя длительность звонка: {service_seconds:.2f} сек")
print(f"Максимальное время ожидания: {max_wait_seconds:.2f} сек")
print(f"ρ: {rho:.8f}")
print(f"Pпр: {P_pr:.10f}")
print(f"Pотк: {P_otk:.10f}")
print(f"P_wait: {P_wait:.10f}")
print(f"Lq: {Lq:.10f}")
print(f"Wq: {Wq_sec:.10f} сек")
print(f"W: {W_sec:.10f} сек")
print(f"Потерянные звонки в час: {lost_calls_per_hour:.10f}")
print(f"Рекомендуемое число сотрудников: {recommended_staff}")
print("\nРЕКОМЕНДАЦИИ:")
for item in recommendations:
    print(f"- {item}")
print("=" * 70)

/tmp/ipykernel_3532/4154479606.py:338: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  out_ws.update("A1", dashboard_rows)


СИСТЕМА МАССОВОГО ОБСЛУЖИВАНИЯ С ОГРАНИЧЕНИЕМ ПО ВРЕМЕНИ ОЖИДАНИЯ
Сотрудников: 3
Звонков в час: 85.0
Средняя длительность звонка: 180.00 сек
Максимальное время ожидания: 60.00 сек
ρ: 1.41666667
Pпр: 0.0221175080
Pотк: 0.3883300025
P_wait: 0.3172292488
Lq: 0.7858608900
Wq: 16.7248100900 сек
W: 196.1940560915 сек
Потерянные звонки в час: 33.0080502152
Рекомендуемое число сотрудников: 7

РЕКОМЕНДАЦИИ:
- Решение: УВЕЛИЧИТЬ ШТАТ до 7 сотрудников
- Причина: высокая доля потерянных звонков
- Система близка к перегрузке
